In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import xgboost as xgb
from sklearn.model_selection import RandomizedSearchCV

In [2]:
# Load the regression-ready dataset
df = pd.read_csv("../../Intermediate/10_regression_ready_dataset.csv")

pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)

print(df.shape)
print(df.dtypes)

(11852520, 63)
cell_start                      str
cell_end                        str
month                         int64
weekend                        bool
time_early_morning             bool
time_evening                   bool
time_midday                    bool
time_morning                   bool
time_night                     bool
ride_count                    int64
pay_1250_or_less_start      float64
early_start_jobs_start      float64
entertainment_jobs_start    float64
age_29_or_younger_start     float64
white_collar_jobs_start     float64
pay_1251_to_3333_start      float64
sex_female_start            float64
job_ct_start                float64
all_day_jobs_start          float64
pay_1250_or_less_end        float64
early_start_jobs_end        float64
entertainment_jobs_end      float64
age_29_or_younger_end       float64
white_collar_jobs_end       float64
pay_1251_to_3333_end        float64
sex_female_end              float64
job_ct_end                  float64
all_day_jobs_

In [5]:
# Define target and feature columns (exclude IDs and target)
id_cols = ['cell_start', 'cell_end']
target_col = 'ride_count'
feature_cols = [c for c in df.columns if c not in id_cols + [target_col]]

X = df[feature_cols]
y = df[target_col]

# Split into train and test sets (80/20)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(X_train.shape, X_test.shape)

(9482016, 60) (2370504, 60)


In [6]:
# Check distribution of the target variable
print(y_train.describe())
print(y_train.value_counts().head(10))

count    9.482016e+06
mean     3.823018e+00
std      8.238143e+00
min      1.000000e+00
25%      1.000000e+00
50%      1.000000e+00
75%      3.000000e+00
max      1.596000e+03
Name: ride_count, dtype: float64
ride_count
1     4755493
2     1642276
3      814913
4      492958
5      330072
6      238731
7      179193
8      139883
9      111776
10      91227
Name: count, dtype: int64


# Baseline: Linear regression на log(ride_count)

In [7]:
# Log-transform the target (ride_count is always >= 1, so log is safe, no need for log1p)
y_train_log = np.log(y_train)
y_test_log = np.log(y_test)

# Fit linear regression
lr_model = LinearRegression()
lr_model.fit(X_train, y_train_log)

# Predict on test set
y_pred_log = lr_model.predict(X_test)

# Evaluate in log space
rmse_log = np.sqrt(mean_squared_error(y_test_log, y_pred_log))
mae_log = mean_absolute_error(y_test_log, y_pred_log)
r2_log = r2_score(y_test_log, y_pred_log)

print(f"RMSE (log space): {rmse_log:.4f}")
print(f"MAE (log space): {mae_log:.4f}")
print(f"R2 (log space): {r2_log:.4f}")

RMSE (log space): 0.7970
MAE (log space): 0.6320
R2 (log space): 0.2536


In [8]:
# Convert predictions back to original scale
y_pred = np.exp(y_pred_log)

rmse = np.sqrt(mean_squared_error(y_test, y_pred))
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f"RMSE (original scale): {rmse:.4f}")
print(f"MAE (original scale): {mae:.4f}")
print(f"R2 (original scale): {r2:.4f}")

RMSE (original scale): 8.0085
MAE (original scale): 2.6106
R2 (original scale): 0.0536


# XGBoost

In [12]:


# Train a baseline XGBoost model with Poisson objective for count data
xgb_model = xgb.XGBRegressor(
    objective='count:poisson',
    n_estimators=100,
    max_depth=6,
    learning_rate=0.1,
    random_state=42,
    n_jobs=-1
)

xgb_model.fit(X_train, y_train)

# Predict on test set
y_pred_xgb = xgb_model.predict(X_test)

rmse_xgb = np.sqrt(mean_squared_error(y_test, y_pred_xgb))
mae_xgb = mean_absolute_error(y_test, y_pred_xgb)
r2_xgb = r2_score(y_test, y_pred_xgb)

print(f"RMSE: {rmse_xgb:.4f}")
print(f"MAE: {mae_xgb:.4f}")
print(f"R2: {r2_xgb:.4f}")

RMSE: 6.4221
MAE: 2.4772
R2: 0.3914


In [13]:
# Get feature importances
importance_df = pd.DataFrame({
    'feature': feature_cols,
    'importance': xgb_model.feature_importances_
}).sort_values('importance', ascending=False)

print(importance_df.head(20))

                    feature  importance
59            grid_distance    0.103115
23               job_ct_end    0.078669
2        time_early_morning    0.078360
1                   weekend    0.077999
47          on_street_start    0.064796
49            on_street_end    0.062303
14             job_ct_start    0.056210
6                time_night    0.030118
20    white_collar_jobs_end    0.027412
11  white_collar_jobs_start    0.025262
57             com_lots_end    0.025117
0                     month    0.023852
54           com_lots_start    0.022213
4               time_midday    0.021933
3              time_evening    0.019995
18   entertainment_jobs_end    0.017624
30         age_15_to_34_end    0.017347
27       age_15_to_34_start    0.016500
19    age_29_or_younger_end    0.016428
17     early_start_jobs_end    0.016418


In [14]:
from sklearn.model_selection import RandomizedSearchCV

# Use a subsample of the training data for faster hyperparameter search
sample_size = 500_000
sample_idx = X_train.sample(n=sample_size, random_state=42).index
X_train_sample = X_train.loc[sample_idx]
y_train_sample = y_train.loc[sample_idx]

# Define parameter distributions to search over
param_dist = {
    'n_estimators': [100, 200, 300],
    'max_depth': [4, 6, 8],
    'learning_rate': [0.05, 0.1, 0.2],
    'min_child_weight': [1, 5, 10],
    'subsample': [0.7, 0.85, 1.0],
    'colsample_bytree': [0.7, 0.85, 1.0]
}

xgb_search_model = xgb.XGBRegressor(
    objective='count:poisson',
    random_state=42,
    n_jobs=-1
)

random_search = RandomizedSearchCV(
    xgb_search_model,
    param_distributions=param_dist,
    n_iter=20,
    scoring='neg_mean_squared_error',
    cv=3,
    random_state=42,
    verbose=2,
    n_jobs=1  # xgb already uses n_jobs=-1 internally
)

random_search.fit(X_train_sample, y_train_sample)

print("Best params:", random_search.best_params_)
print("Best CV score (neg MSE):", random_search.best_score_)

Fitting 3 folds for each of 20 candidates, totalling 60 fits
[CV] END colsample_bytree=0.85, learning_rate=0.2, max_depth=8, min_child_weight=5, n_estimators=100, subsample=0.7; total time=   1.2s
[CV] END colsample_bytree=0.85, learning_rate=0.2, max_depth=8, min_child_weight=5, n_estimators=100, subsample=0.7; total time=   1.2s
[CV] END colsample_bytree=0.85, learning_rate=0.2, max_depth=8, min_child_weight=5, n_estimators=100, subsample=0.7; total time=   1.3s
[CV] END colsample_bytree=0.7, learning_rate=0.1, max_depth=8, min_child_weight=5, n_estimators=200, subsample=0.85; total time=   2.3s
[CV] END colsample_bytree=0.7, learning_rate=0.1, max_depth=8, min_child_weight=5, n_estimators=200, subsample=0.85; total time=   2.2s
[CV] END colsample_bytree=0.7, learning_rate=0.1, max_depth=8, min_child_weight=5, n_estimators=200, subsample=0.85; total time=   2.2s
[CV] END colsample_bytree=0.85, learning_rate=0.05, max_depth=8, min_child_weight=1, n_estimators=200, subsample=1.0; total

Best parameters:

max_depth=8, learning_rate=0.2, n_estimators=300, subsample=0.85, colsample_bytree=1.0, min_child_weight=1.

In [15]:
# Train final model with best params on the FULL training set
best_params = random_search.best_params_

xgb_final = xgb.XGBRegressor(
    objective='count:poisson',
    random_state=42,
    n_jobs=-1,
    **best_params
)

xgb_final.fit(X_train, y_train)

y_pred_final = xgb_final.predict(X_test)

rmse_final = np.sqrt(mean_squared_error(y_test, y_pred_final))
mae_final = mean_absolute_error(y_test, y_pred_final)
r2_final = r2_score(y_test, y_pred_final)

print(f"RMSE: {rmse_final:.4f}")
print(f"MAE: {mae_final:.4f}")
print(f"R2: {r2_final:.4f}")

RMSE: 4.2273
MAE: 1.8719
R2: 0.7363
